<a href="https://colab.research.google.com/github/adrinorosario/legal-pragmatic-inference/blob/main/hybrid_clause_and_term_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Clause and Vague Term Extraction

Working on this approach here, the goal is to extract the contractual clauses and the underlying vague terms in the particular clauses. For this, we will be used 3 datasets:

*   [CUAD](https://huggingface.co/datasets/theatticusproject/cuad)
*   [ContractNLI](https://huggingface.co/datasets/kiddothe2b/contract-nli); extensive information can be found [here](https://stanfordnlp.github.io/contract-nli/). Refer to the paper [here](https://arxiv.org/pdf/2110.01799)
*   [LEDGAR](https://huggingface.co/datasets/coastalcph/lex_glue) (the subset in LexGLUE)

Extracting the anchor clauses and the vague terms will form the first two parts of the [triplet dataset](https://github.com/adrinorosario/legal-pragmatic-inference/blob/main/docs/research/dataset_construction.md).




In [ ]:
from IPython.display import HTML, display

def set_css():
    display(HTML('''
    <style>
        pre {
            white-space: pre-wrap;       /* CSS3 */
            white-space: -moz-pre-wrap;  /* Mozilla, since 1999 */
            white-space: -pre-wrap;      /* Opera 4-6 */
            white-space: -o-pre-wrap;    /* Opera 7 */
            word-wrap: break-word;       /* Internet Explorer 5.5+ */
        }
    </style>
    '''))

# Swapped 'pre_run' with 'pre_execute' to match modern IPython specifications
get_ipython().events.register('pre_execute', set_css)

In [ ]:
import json
import requests
import re
import copy

import torch
from tqdm.auto import tqdm

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
  HF_TOKEN = userdata.get('HF_TOKEN')
  login(token=HF_TOKEN)
  print("Successfully logged into Hugging Face.")
except Exception as e:
  print("Error: Ensure you have added your 'HF_TOKEN' to the Colab Secrets manager (key icon on the left).")

Successfully logged into Hugging Face.


## Extraction from CUAD

For this, we are using the JSON file from the HuggingFace page of the CUAD Dataset which can be found [here](https://huggingface.co/datasets/theatticusproject/cuad/tree/main/CUAD_v1)

In [ ]:
# read the json file
cuad_v1_json = "/content/CUAD_v1.json"

with open(cuad_v1_json, "r") as file:
  data = json.load(file)

data.keys()

dict_keys(['version', 'data'])

In [ ]:
len(data["data"]) # contains 510 entries

510

In [ ]:
cuad_data = data["data"]
cuad_data[0].keys() # each entry, i.e., a contract contains the title and the paragraphs in it

dict_keys(['title', 'paragraphs'])

In [ ]:
print(f"Type of cuad_data[0]['paragraphs']: {type(cuad_data[0]["paragraphs"])}")
print(f"Length of cuad_data[0]['paragraphs]: {len(cuad_data[0]["paragraphs"])}")

print(f"\nType of cuad_data[0]['paragraphs'][0]: {type(cuad_data[0]['paragraphs'][0])}")
print(f"Length of cuad_data[0]['paragraphs'][0]: {len(cuad_data[0]['paragraphs'][0])}")
print(f"Keys of cuad_data[0]['paragraphs'][0]: {cuad_data[0]["paragraphs"][0].keys()}")

Type of cuad_data[0]['paragraphs']: <class 'list'>
Length of cuad_data[0]['paragraphs]: 1

Type of cuad_data[0]['paragraphs'][0]: <class 'dict'>
Length of cuad_data[0]['paragraphs'][0]: 2
Keys of cuad_data[0]['paragraphs'][0]: dict_keys(['qas', 'context'])


1.   You are accessing each contract's paragraphs, which is a **list**.
2.   Each list of paragraphs contains a dictionary, which has two keys: **qas** and **context**



In [ ]:
print(f"Type of cuad_data[0]['paragraphs'][0]['qas']: {type(cuad_data[0]['paragraphs'][0]['qas'])}")
print(f"Length of cuad_data[0]['paragraphs'][0]['qas']: {len((cuad_data[0]['paragraphs'][0]['qas']))}")

print("\nLooking at a single qas:")
print(cuad_data[0]['paragraphs'][0]['qas'][0])
print(f"Keys in a single qas: {cuad_data[0]['paragraphs'][0]['qas'][0].keys()}")

Type of cuad_data[0]['paragraphs'][0]['qas']: <class 'list'>
Length of cuad_data[0]['paragraphs'][0]['qas']: 41

Looking at a single qas:
{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}], 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'is_impossible': False}
Keys in a single qas: dict_keys(['answers', 'id', 'question', 'is_impossible'])


In [ ]:
cuad_data[0]['paragraphs'][0]['qas'][4]["answers"][0]

{'text': 'The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distributor  the                            last Sample, as defined  hereinafter.',
 'answer_start': 5268}

1.   **qas** is a list. Each item in the list is a dictionary.
2.   A single item in the **qas** list has the following keys: **answers**, **id**, **question**, and **is_impossible**.



In [ ]:
cuad_data[0]['paragraphs'][0].keys()

dict_keys(['qas', 'context'])

In [ ]:
# inspect the first contract's paragraph
cuad_data[0]["paragraphs"][0].keys() # each paragraph contains 'qas' and 'context'

# inspect the qas first; inspect the first item in the list
cuad_data[0]["paragraphs"][0]["qas"][0] # each qas item contains 'answers' which is a list of its own, 'id', 'question', and 'is_impossible'

# focussing on a single paragraph
single_paragraph = cuad_data[0]["paragraphs"][0]
# focussing on the same anchor text
anchor_text = single_paragraph["context"]

# loop through all questions inside the single paragraph
for qa in single_paragraph["qas"]:
  # skip categories where no terms were found
  if qa["is_impossible"]:
    continue

  clause_id = qa["id"].split("__")[-1]
  question = qa["question"]
  answers = [ans["text"] for ans in qa["answers"]]

  # print out the findings
  print(f"ID: {clause_id}")
  print(f"QUESTION: {question}")
  print(f"ANSWER: {answers}")
  print("="*40)

ID: Document Name
QUESTION: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
ANSWER: ['DISTRIBUTOR AGREEMENT']
ID: Parties
QUESTION: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
ANSWER: ['Distributor', 'Electric City Corp.', 'Electric City of Illinois L.L.C.', 'Company', 'Electric City of Illinois LLC']
ID: Agreement Date
QUESTION: Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
ANSWER: ['7th day of September, 1999.']
ID: Effective Date
QUESTION: Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. Details: The date when the contract is effective 
ANSWER: ['The term of this  Agreement  shall be ten (10)                        

In [ ]:
anchor_text

'EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.\n\n                                    RECITALS\n\n         A. The  Company\'s  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.\n\n         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has or  will  hav

In [ ]:
# extract the unique clauses from the contracts
unique_clauses = set()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]
      unique_clauses.add(clause_id)

print("Unique clauses found in the first 5 contracts:")
print(unique_clauses)
print(f"Number of unique clauses found: {len(unique_clauses)}")

Unique clauses found in the first 5 contracts:
{'Document Name', 'Revenue/Profit Sharing', 'Third Party Beneficiary', 'Renewal Term', 'Irrevocable Or Perpetual License', 'Joint Ip Ownership', 'Affiliate License-Licensor', 'Uncapped Liability', 'Non-Disparagement', 'Notice Period To Terminate Renewal', 'Affiliate License-Licensee', 'Warranty Duration', 'Parties', 'Unlimited/All-You-Can-Eat-License', 'Audit Rights', 'Cap On Liability', 'Price Restrictions', 'Effective Date', 'Volume Restriction', 'Competitive Restriction Exception', 'Exclusivity', 'Post-Termination Services', 'Agreement Date', 'Rofr/Rofo/Rofn', 'No-Solicit Of Employees', 'Most Favored Nation', 'Expiration Date', 'Anti-Assignment', 'License Grant', 'Ip Ownership Assignment', 'Source Code Escrow', 'Covenant Not To Sue', 'Governing Law', 'Change Of Control', 'No-Solicit Of Customers', 'Insurance', 'Liquidated Damages', 'Non-Compete', 'Termination For Convenience', 'Minimum Commitment', 'Non-Transferable License'}
Number of 

In [ ]:
# HIGH PRIORITY CLAUSES THAT ARE SUBJECT TO INTENSE PRAGMATIC INFERENCE
# AND LITIGATION IN COURTS
tier_1_clauses = {
    'Audit Rights',
    'Termination For Convenience',
    'Most Favored Nation',
    'Non-Compete',
    'Insurance',
    'Minimum Commitment',
    'Post-Termination Services',
    'Warranty Duration'
}

# STRICT PROHIBITIONS BUT CAN ALSO BE LITIGATED IN COURTS BASED ON THE
# CONTEXT OF THE CONTRACT AND CASE
tier_2_clauses = {
    'Exclusivity',
    'Anti-Assignment',
    'Cap On Liability',
    'Competitive Restriction Exception',
    'Covenant Not To Sue',
    'No-Solicit Of Customers',
    'No-Solicit Of Employees',
    'Non-Disparagement',
    'Non-Transferable License',
    'Revenue/Profit Sharing',
    'Rofr/Rofo/Rofn',
    'Source Code Escrow',
    'Third Party Beneficiary',
    'Price Restrictions',
    'License Grant',
    'Affiliate License-Licensor',
    'Affiliate License-Licensee',
    'Ip Ownership Assignment',
    'Joint Ip Ownership',
    'Irrevocable Or Perpetual License',
    'Unlimited/All-You-Can-Eat-License',
    'Volume Restriction'
}

# DETERMINISTIC OPERATIONS AND CLAUSES; DATES, AMOUNT, NUMERICAL VALUES, AND
# OTHER CLEAR DETERMINISTIC OPERATIONS
tier_3_clauses = {
    'Agreement Date',
    'Document Name',
    'Effective Date',
    'Expiration Date',
    'Governing Law',
    'Liquidated Damages',
    'Notice Period To Terminate Renewal',
    'Renewal Term',
    'Parties',
    'Uncapped Liability'
}

In [ ]:
vagueness_seed_set = {
    # ── Effort & diligence ──────────────────────────────────────────────
    "reasonable efforts", "best efforts", "commercially reasonable",
    "due diligence", "reasonable care", "good faith", "workmanlike manner",
    "best practices", "reasonable endeavours", "all reasonable steps",
    "every reasonable effort", "diligent efforts", "reasonable commercial efforts",
    "utmost care", "exercise of judgment", "commercially practicable",
    "economically reasonable", "technically feasible",
    "consistent with good industry practice", "as would a prudent operator",
    "acting reasonably", "using its discretion",

    # ── Time & urgency ──────────────────────────────────────────────────
    "promptly", "in a timely manner", "as soon as practicable",
    "without undue delay", "for a reasonable period",
    "termination of this Agreement", "from time to time", "periodic",
    "duration", "seasonable", "business hours", "within a reasonable time",
    "with all due speed", "expeditiously", "at the earliest opportunity",
    "without unnecessary delay", "within a commercially reasonable period",
    "in due course", "forthwith", "in due time", "on a timely basis",
    "in the near term", "shortly after", "when practicable",
    "upon reasonable notice", "reasonable notice period",

    # ── Scope, degree & quantity ────────────────────────────────────────
    "material", "substantial", "limited", "relevant", "related", "generally",
    "appropriate", "similar", "de minimis", "significant", "incidental",
    "including but not limited to", "inter alia", "and/or",
    "save as otherwise provided", "appreciable", "meaningful", "non-trivial",
    "measurable", "proportionate", "commensurate", "reasonably proportionate",
    "unduly burdensome", "reasonably necessary", "to the extent practicable",
    "to a reasonable extent", "without limitation", "as applicable",
    "where relevant", "as appropriate", "to the extent required",

    # ── Harm, change & threshold ────────────────────────────────────────
    "material adverse effect", "material breach", "material adverse change",
    "material adverse impact", "material adverse consequence",
    "materially and adversely", "substantial impairment", "material disruption",
    "material deviation", "materially prejudice", "disproportionate impact",
    "unreasonable hardship", "undue prejudice", "undue harm", "undue risk",

    # ── Necessity & discretion ──────────────────────────────────────────
    "necessary", "sole discretion", "need to know", "confidential nature",
    "adequate", "satisfactory", "proper", "intended purpose",
    "not to be unreasonably withheld", "mutual satisfaction", "at its option",
    "consultation", "absolute discretion", "unfettered discretion",
    "not to be unreasonably delayed", "not to be unreasonably conditioned",
    "without arbitrary restriction", "reasonably required",
    "reasonably requested", "if deemed appropriate", "as deemed necessary",
    "in its reasonable opinion", "acting in good faith",
    "in its reasonable judgment", "as it sees fit", "as directed",

    # ── Industry norms & quality ────────────────────────────────────────
    "customary", "ordinary course of business", "industry standard",
    "standard practice", "normally", "comparable", "acceptable",
    "conventional", "fit for purpose", "first-class condition",
    "commercially sensitive", "prevailing market practice",
    "generally accepted practice", "market standard",
    "accepted industry norms", "standard market terms",
    "customary market conditions", "in accordance with accepted methods",
    "consistent with past practice", "as is customary",
    "in accordance with best available techniques",
    "reasonable engineering standards", "professionally acceptable",
    "to a professional standard", "of merchantable quality",
    "of satisfactory quality",

    # ── Knowledge, intent & foresight ──────────────────────────────────
    "foreseeable", "contemplated", "intended", "anticipated", "applicable",
    "knowledge", "directly or indirectly", "disclosed in confidence",
    "all copies", "survive", "mutual agreement", "substantially similar",
    "reasonable expectations", "actual knowledge", "constructive knowledge",
    "reasonably should have known", "to the best of its knowledge",
    "as far as it is aware", "reasonably foreseeable",
    "unforeseen circumstances", "unanticipated events",
    "beyond reasonable expectation", "reasonable belief", "bona fide belief",
    "reasonable grounds", "having regard to all circumstances",

     # ── Confidentiality & information ───────────────────────────────────
    "proprietary information", "non-public information",
    "sensitive business information", "trade secrets",
    "sufficiently confidential", "reasonably considered confidential",
    "maintained in confidence", "treated as confidential",
    "in accordance with confidentiality obligations",

    # ── Financial & commercial terms ────────────────────────────────────
    "commercially attractive", "economically viable",
    "commercially justifiable", "at a reasonable price", "fair market value",
    "arm's length", "at prevailing rates", "on reasonable commercial terms",
    "on competitive terms", "at a rate reflecting market conditions",
    "at cost", "without unreasonable mark-up", "reasonable compensation",
    "reasonable fees",

    # ── Survival, agreement & modification ─────────────────────────────
    "notwithstanding the foregoing", "without prejudice to",
    "subject to the foregoing", "except as otherwise agreed",
    "unless otherwise specified", "where not inconsistent", "insofar as",
    "to the fullest extent permitted by law", "as may be amended",
    "as modified from time to time", "by mutual written consent",
}

len(vagueness_seed_set)

206

### Extracting clause categories and texts

Using the dataset available, the following needs to be extracted:

*   Document ID (for cross referencing later if needed)
*   Clause category/ID
*   Text from the clause that describes the contract
*   The tier it belongs to
*   The set of vague terms contained in it

Furthermore, we also need to check whether the clause is:

*   Lethal - belongs to tier 1
*   Has context risk - belongs to tier 2

All of these will be stored as a dictionary, and housed in a list



In [ ]:
clause_and_terms = list()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      # check_lethality flags if the clause belongs to tier 1
      # context_risk flags if the clause belongs to tier 2
      check_lethality, context_risk = False, False
      tier = 0

      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]

      # check if the clause id belongs to tier 3, if yes, discard it
      if clause_id in tier_3_clauses:
        continue

      # check tier and assign label
      if clause_id in tier_1_clauses:
        tier = 1
      elif clause_id in tier_2_clauses:
        tier = 2

      clause_text = qa["answers"][0]["text"]
      contract_id = contract['title']
      vague_terms = {term for term in vagueness_seed_set if term in clause_text}


      # check the lethality of the clause
      if tier == 1 and vague_terms:
        check_lethality = True
        context_risk = False
      elif tier == 2 and vague_terms:
        check_lethality = False
        context_risk = True

      if check_lethality or context_risk:
        clause_term_dict = {
            "contract_id": contract_id,
            "clause_category": clause_id,
            "clause_text": clause_text,
            "tier": tier,
            "vague_terms": vague_terms,
            "is_lethal": check_lethality,
            "has_context_risk": context_risk
        }

        clause_and_terms.append(clause_term_dict)

In [ ]:
print(f"Extracted data: {len(clause_and_terms)}")

Extracted data: 1428


In [ ]:
from prompt_toolkit.shortcuts import print_container
import random

sampling_size = int(len(clause_and_terms) * 0.20)
compressed_sampling_size = int(sampling_size * 0.05)

for i in range(compressed_sampling_size):
  random_idx = random.randint(0, len(clause_and_terms)-1)

  print(f"CLAUSE CATEGORY: {clause_and_terms[random_idx]["clause_category"]}")
  print(f"CLAUSE TEXT: {clause_and_terms[random_idx]["clause_text"]}")
  print(f"TIER: {clause_and_terms[random_idx]["tier"]}")
  print(f"VAGUE TERMS: {clause_and_terms[random_idx]["vague_terms"]}")
  print(f"IS LETHAL: {clause_and_terms[random_idx]["is_lethal"]}")
  print(f"HAS CONTEXT RISK: {clause_and_terms[random_idx]["has_context_risk"]}")
  print("="*40, end="\n\n")

CLAUSE CATEGORY: Non-Transferable License
CLAUSE TEXT: Such rights to Joint Improvements shall be solely for use by the Company and shall not be transferable to any Third Party except in connection with a merger, consolidation, or the sale or transfer of substantially all of Company's assets associated with performance under this IP Agreement.
TIER: 2
VAGUE TERMS: {'substantial'}
IS LETHAL: False
HAS CONTEXT RISK: True

CLAUSE CATEGORY: Rofr/Rofo/Rofn
CLAUSE TEXT: Licensee's           exercise of the Option is at its sole discretion.  Licensee may           exercise the Option by written notice to Licensor and Skunkware at any           time during the Option Period.
TIER: 2
VAGUE TERMS: {'sole discretion'}
IS LETHAL: False
HAS CONTEXT RISK: True

CLAUSE CATEGORY: Non-Transferable License
CLAUSE TEXT: Without  Party A's permission in writing, Party B may not disclose or sublicense the Program Content to any third party, except for the Program  Content related to Party B Business.
TIER:

Right now, each data point `clause_and_terms` houses the vague clauses, vague terms, and the accompanying signals such as lethality and context risks.

From this, we move on to semantic mapping with the case law from the Harvard Corpus, i.e., [COLD Cases](https://huggingface.co/datasets/harvard-lil/cold-cases).

The `clause_text` present in each data point will be used to semantically map the right case law from this corpus which will be the judicial prose of the triplet.



## Semantic Mapping with Harvard LIL COLD Cases

Taking `clause_and_terms`, we will use the `clause_text` key/item and encode it into its **vector embeddings**.

A judge will not always write the contractual clauses as and how it appears in a contract in his reasoning; hence, string matching will fail. We need to map them in a dense **vector space**.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# intialise the asymmetric Gemma embedding model
bi_encoder = SentenceTransformer("google/embeddinggemma-300m").to("cuda")

# vectorize the clause_texts in each data point in clause_and_terms
anchor_clause_texts = [data_point["clause_text"] for data_point in clause_and_terms]

anchor_clause_embeddings = bi_encoder.encode(
    anchor_clause_texts,
    prompt="task: search result| query: ",
    show_progress_bar=True,
    batch_size=32,
    convert_to_tensor=True
    )

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

## Dual-Vector Anchoring

### Implementing a coarse filter on harvard-lil/cold-cases

The coarse filter is not looking for contratual obligations here. Rather, it's focus is to filter out all sentences from the opinion texts that are not useful for the vector embeddings to compute similarity with.

From a micro perspective, the filter will:

*   Check if the opinion texts belong to **contract, commercial law, corporate law, or other such similar cases.** No statutory, administrative, or tort law cases will be considered.
*   Segment each opinion text into individual sentences. Use sentences that have $> 10$ and $\le 100$ words in a sentence. This can be tuned depending on the quality of the filter's output.
*   Ensure **no statutory sentences** are present. *This is important.*
*   No checking for modals or seed words (vague terms) in this step.





In [ ]:
# Statutory signals
STATUTORY_PATTERN = re.compile(
    r'\b(§|Code|Section|Article|Constitution|Statute|Act of \d{4}|'
    r'U\.S\.C\.|W\.Va\. Code|provided by law|mandates that|'
    r'every municipality|any person|no employee|all employers)\b',
    re.IGNORECASE
)

In [ ]:
# structural indicators for retrospective reasoning prose
REASONING_INDICATORS = re.compile(
  # Epistemic / judgment verbs (strong signals)
  r'\b(held|concluded|found|determined|argued|asserted|interpreted|appealed|testified)\b|'
  r'\b(finds|found|ruled|decided|opined|considered|recognised|acknowledged)\b|'
  r'\b(construed|construction|construing)\b|'

  # First-person judicial voice (strong signals)
  r'\b(I\s+find|I\s+am\s+satisfied|I\s+consider|I\s+prefer|I\s+accept|I\s+reject)\b|'
  r'\b(I\s+agree|I\s+disagree|I\s+conclude|I\s+hold|in\s+my\s+judgment|in\s+my\s+view)\b|'
  r'\b(the\s+court\s+finds|the\s+court\s+held|the\s+court\s+concludes|the\s+court\s+considers)\b|'

  # Plain meaning / interpretive analysis (strong signals)
  r'\b(on\s+a\s+proper\s+construction|the\s+plain\s+meaning|the\s+natural\s+meaning)\b|'
  r'\b(purposive|contextual\s+reading|read\s+as\s+a\s+whole|read\s+together)\b|'
  r'\b(the\s+better\s+view|the\s+correct\s+interpretation|properly\s+construed)\b|'

  # Inferential / consequential connectors (medium signals)
  r'\b(therefore|accordingly|it\s+follows|consequently|necessarily\s+means)\b|'
  r'\b(it\s+is\s+clear\s+that|it\s+must\s+follow|this\s+suggests|this\s+indicates)\b|'
  r'\b(would\s+have|could\s+have|should\s+have|must\s+have)\b|'

  # Reasonableness / obligation framing (medium signals)
  r'\b(reasonable|reasonably|unreasonable|unreasonably)\b|'
  r'\b(the\s+parties\s+intended|the\s+intention\s+of\s+the\s+parties|objectively\s+construed)\b|'
  r'\b(implied\s+term|implied\s+obligation|necessary\s+implication)\b|'

  # Comparative / distinguishing reasoning (medium signals)
  r'\b(unlike|by\s+contrast|whereas|distinguished\s+from|analogous\s+to)\b|'
  r'\b(consistent\s+with|inconsistent\s+with|contrary\s+to)\b|'

  # Original weak signals retained
  r'\b(this\s+provision|such\s+obligation|the\s+disputed|underlying\s+contract)\b',

  re.IGNORECASE
)

In [ ]:
def filter_cold_cases_opinion_text(
    opinion_text: str,
    window_size: int = 3,
    step_size: int = 1
    ):
  """
  """

  # store all valid sentences individually, not as an entire opinion text
  clause_sentences = []
  reasoning_sentences = []

  # split the entire opinion text into individual sentences
  sentences = re.split(r'(?<=[.!?])\s+', opinion_text)

  for sentence in sentences:
    sentence = sentence.strip() # strip off all leading and trailing whitespaces

    # check for statutory patterns and skip them
    if STATUTORY_PATTERN.search(sentence):
      continue

    # check the number of words in the sentence and eliminate accordingly
    words = sentence.split()
    if len(words) < 10 or len(words) > 100:
      continue

    # the gatekeeper routing to embedding or RRL
    if REASONING_INDICATORS.search(sentence):
      reasoning_sentences.append(sentence)
    else:
      clause_sentences.append(sentence)

  # check if valid sentences is not empty, and construct the sliding windows
  if clause_sentences:

    # construct the overlapping sliding windows for the clause track
    sliding_windows = []
    num_sentences = len(clause_sentences)

    if num_sentences >= window_size:
      # loop from the starting of the sentences list
      # up until the last window_size element+1
      # step the control by step size
      # this creates the sliding window
      for i in range(0, num_sentences - window_size + 1, step_size):
        # join window_size sentences into one text block
        window_text = " ".join(clause_sentences[i: i + window_size])
        sliding_windows.append({
            "text_window": window_text,
            "start_idx": i,
            "end_idx": i + window_size
        })

    # valid_sentences_embeddings = embedding_model.encode(
    #     clause_sentences,
    #     show_progress_bar=False,
    #     batch_size=64
    # )

    # check the cosine similarity with the cuad anchors
    # similarities = embedding_model.similarity(
    #     anchor_clause_embeddings,
    #     valid_sentences_embeddings
    # )

    output = {
        "opinion_text": opinion_text,
        "sentences": clause_sentences,
        "clause_sliding_windows": sliding_windows,
        "reasoning_sentences": reasoning_sentences
    }

    return output

  else:
    return False

In [ ]:
from datasets import load_dataset

# load the dataset from HuggingFace: https://huggingface.co/datasets/harvard-lil/cold-cases
cold_cases = load_dataset(
    "harvard-lil/cold-cases",
    split="train",
    streaming=True
)

cold_cases["opinions"]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
# parse the dataset and check for all the different case natures
case_natures = set()

# get all the different natures of cases
for idx, case in enumerate(cold_cases):
  if idx >= 50000:
    break
  nature = case.get("nature_of_suit", "") or "" # read the nature of the case
  case_natures.add(nature)

len(case_natures), case_natures

KeyboardInterrupt: 

In [ ]:
target_case_natures = {
    'Tort, Contract, and Real Property',
    'Private Civil Diversity',
    'Private Civil Federal',
    'OIL, GAS AND MINERALS',
}

In [ ]:
count = 0

for case in cold_cases:
  if count >= 10:
    break

  nature = case.get("nature_of_suit", "") or "" # read the nature of the case
  # only proceed with the targetted case natures
  if nature in target_case_natures:

      opinions = case.get("opinions", []) # extract the opinions into a list

      # check if the case has an opinion text that can be extracted
      if opinions and opinions[0].get("opinion_text"):
        validity_status = filter_cold_cases_opinion_text(opinions[0]['opinion_text'])

        # if the filter returns false, skip document
        if not validity_status:
          continue

        # here now, we need to isolate the high scoring coordinates
        # if the cosine similarity >= 70
        # pair the raw case sentence text directly with the metadata of the
        # matching CUAD anchor clause
        row_indices, col_indices = torch.where(validity_status['similarities'] >= 0.70)

        for row, col in zip(row_indices, col_indices):
          row_idx = row.item()
          col_idx = col.item()

          # extract the matching score from the matrix
          score = validity_status['similarities'][row_idx][col_idx].item()

          # obtain the matching cuad metadata from the json directly
          # row_idx looks up the cuad anchors
          matching_cuad_metadata = clause_and_terms[row_idx]

          # col_idx looks up the raw sentence from the valid sentences
          matching_raw_sentence = validity_status["sentences"][col_idx]

          print(f"Matching Score: {score: .2f}")
          print(f"|-- CUAD CATEGORY: {matching_cuad_metadata["clause_category"]}")
          print(f"|-- CUAD Anchor: {matching_cuad_metadata["clause_text"]}")
          print(f"└── Raw sentence: {matching_raw_sentence}")
          print("="*50)

          count += 1

KeyboardInterrupt: 

Check the nature of the cosine similarity of outputs for following thresholds:

*   $> 60\% \text{ and } \le 70\%$
*   $> 70\% \text{ and } \le 75\%$
*   $> 75\%$





**Notes until this point in the extraction:**

1.  The threshold bucket that contains scores higher than 75% contains pure lexical matchings which are similar to the substring matching that can be achieved using pure code
2.  The bucket with scorings in the 70-75% range house a lot of text is judicial reasoning, rather than the clauses.
    *   The sentences assertive, stating the judge's interpretation of the clauses and its obligations, **not** the clause itself
    *   These are mappings that are needed for the interpretation part of the triplet structure which will what the reward system will rely on
    *   Failure to isolate these two aspects will corrupt the learning process of the reinforcement learning loop
3.  Surprisingly, the 60-70% bucket contains positive signals where it had identified clauses, some of then including:

    ```
    |-- CUAD CATEGORY: Termination For Convenience
    |-- CUAD Anchor: Either party may, at its option, terminate this Agreement       without cause, effective at any time after January 31, 1999, upon giving       at least ninety (90) days prior written notice of such termination to the       other party.
    └── Raw sentence: Rather, the Advisory Agreements provided for termination without cause upon sixty days’ written notice.
    ```

    This shows promising results. It also contains a lot of structural noise compared to the other two buckets.

In [ ]:
STRUCTURAL_ANCHORS = re.compile(
    r'\b(agreement|section|article|clause|provision|policy|schedule|exhibit|hereunder|parties)\b',
    re.IGNORECASE
)

# Target the non-commercial domains dominating your false positives
DOMAIN_EXCLUSION = re.compile(
    r'\b(medical|malpractice|panel|doctor|patient|decedent|spouse|widow|estate|elective\s+share|divorce|testimony|criminal|guilty)\b',
    re.IGNORECASE
)

In [ ]:
distributed_threshold_results = {
    0.40: [],
    "reasoning_sentences": []
}

# Configuration
threshold = 0.40
target_matches = float('inf')

# Dual progress bars
matches_pbar = tqdm(total=None, desc="Matches Found", unit="match", position=0, leave=True)
cases_pbar = tqdm(desc="Cases Scanned", unit="case", position=1, leave=True)
cases_processed = 0

for case in cold_cases:
    cases_processed += 1
    cases_pbar.update(1)

    nature = case.get("nature_of_suit", "") or ""
    if nature not in target_case_natures:
        continue

    opinions = case.get("opinions", [])
    if not (opinions and opinions[0].get("opinion_text")):
        continue

    opinion_text = opinions[0]["opinion_text"]
    case_data = filter_cold_cases_opinion_text(opinion_text, window_size=3, step_size=1)

    if case_data:
        distributed_threshold_results["reasoning_sentences"].extend(case_data["reasoning_sentences"])

        if case_data["clause_sliding_windows"]:
            # local sentence index tracker per case to eliminate duplication
            used_sentence_indices = set()

            text_blocks = [window["text_window"] for window in case_data["clause_sliding_windows"]]

            window_embeddings = bi_encoder.encode(
                text_blocks,
                prompt="title: none| text: ",
                show_progress_bar=False,
                convert_to_tensor=True
            )

            similarities = bi_encoder.similarity(anchor_clause_embeddings, window_embeddings)

            # Extract only those where score >= threshold
            row_indices, column_indices = torch.where(similarities >= threshold)

            if len(row_indices) == 0:
              continue

            # a document-level lookup to map how many times a window fires across categories
            window_to_categories_map = {}

            for row, col in zip(row_indices, column_indices):
                r_idx = row.item()
                c_idx = col.item()
                cat = clause_and_terms[r_idx]["clause_category"]

                if c_idx not in window_to_categories_map:
                    window_to_categories_map[c_idx] = set()
                window_to_categories_map[c_idx].add(cat)

            for row, col in zip(row_indices, column_indices):
                row_idx = row.item()
                col_idx = col.item()
                score = similarities[row_idx][col_idx].item()

                if score >= threshold:
                    # Enforce the Cross-Category Suppression Gate
                    if len(window_to_categories_map[col_idx]) > 2:
                        continue

                    matching_cuad_metadata = clause_and_terms[row_idx]
                    matching_raw_window_dict = case_data["clause_sliding_windows"][col_idx]
                    matching_raw_window = text_blocks[col_idx]

                    if DOMAIN_EXCLUSION.search(matching_raw_window):
                      continue
                    # enforce structural deontic gate to check the text window
                    if not STRUCTURAL_ANCHORS.search(matching_raw_window):
                      continue

                    start_idx = matching_raw_window_dict["start_idx"]
                    end_idx = matching_raw_window_dict["end_idx"]
                    window_range = set(range(start_idx, end_idx))

                    if len(window_range.intersection(used_sentence_indices)) > 1:
                      continue

                    distributed_threshold_results[threshold].append({
                        "cuad_category": matching_cuad_metadata["clause_category"],
                        "cuad_anchor": matching_cuad_metadata["clause_text"],
                        "raw_sentence": matching_raw_window,
                        "bi_encoder_score": score,
                        "contract_id": case.get("title", "Unknown_Contract")
                    })

                    used_sentence_indices.update(window_range)
                    matches_pbar.update(1)

matches_pbar.close()
cases_pbar.close()
print(f"\nFull Run Complete. Scanned {cases_processed} cases to find {len(distributed_threshold_results[threshold])} matches.")

Matches Found: 0match [00:00, ?match/s]

Cases Scanned: 0case [00:00, ?case/s]

KeyboardInterrupt: 

In [ ]:
len(distributed_threshold_results[0.40]), len(distributed_threshold_results["reasoning_sentences"]), type(distributed_threshold_results[0.40]), type(distributed_threshold_results["reasoning_sentences"])

(5130, 46057, list, list)

In [ ]:
# The file path where you want to save the data
output_path = 'extracted_triplets_6m.json'

# Open the file in write mode and dump the specific 0.40 results list
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(distributed_threshold_results[0.40], f, indent=4, ensure_ascii=False)

print(f"Successfully saved {len(distributed_threshold_results[0.40])} records to {output_path}")

Successfully saved 5130 records to extracted_triplets_6m.json


In [ ]:
reasoning_output_path = 'extracted_reasoning_sentences_6m.json'

with open(reasoning_output_path, 'w', encoding='utf-8') as f:
    json.dump(distributed_threshold_results['reasoning_sentences'], f, indent=4, ensure_ascii=False)

print(f"Successfully saved {len(distributed_threshold_results['reasoning_sentences'])} reasoning sentences to {reasoning_output_path}")

Successfully saved 46057 reasoning sentences to extracted_reasoning_sentences_6m.json


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# save the triplets (this part worked as it is a list of dicts)
triplet_table_6m = pa.Table.from_pylist(distributed_threshold_results[0.40])
pq.write_table(triplet_table_6m, "extracted_triplets_6m.parquet", compression="zstd")

# fix for reasoning sentences: wrap strings in a dictionary to create a column
# we transform ["string1", "string2"] into [{"text": "string1"}, {"text": "string2"}]
reasoning_data_as_dicts = [{"reasoning_prose": s} for s in distributed_threshold_results["reasoning_sentences"]]
reasoning_table_6m = pa.Table.from_pylist(reasoning_data_as_dicts)
pq.write_table(reasoning_table_6m, "extracted_reasoning_sentences_6m.parquet", compression="zstd")

print("Both files successfully saved to Parquet using ZSTD compression.")

Both files successfully saved to Parquet using ZSTD compression.


In [ ]:
# Define the starting point
START_RECORD = 6000000

# Re-initialize or apply skip to the streaming dataset
# This bypasses the first 6 million records
cold_cases_skipped = cold_cases.skip(START_RECORD)

print(f"Resuming scan from record {START_RECORD}...")

# Your loop would then use the skipped object:
# for case in cold_cases_skipped:
#     ...

In [ ]:
if len(distributed_threshold_results[threshold]) > 0:
    num_to_sample = min(7, len(distributed_threshold_results[threshold]))
    for i in range(num_to_sample):
        rand_idx = random.randint(0, len(distributed_threshold_results[threshold]) - 1)
        data_point = distributed_threshold_results[threshold][rand_idx]
        print(f"|-- SCORE: {data_point['bi_encoder_score']:.4f}")
        print(f"|-- CUAD CATEGORY: {data_point['cuad_category']}")
        print(f"|-- CUAD Anchor: {data_point['cuad_anchor']}")
        print(f"└── Raw sentence: {data_point['raw_sentence']}")
        print("="*50, end="\n\n")
else:
    print(f"No results found for threshold {current_threshold}.")

|-- SCORE: 0.4218
|-- CUAD CATEGORY: Cap On Liability
|-- CUAD Anchor: of such damages. in any event, other than claims covered by the next sentence, the liability of BNL to VIP for any reason and upon any cause of action and claim in contract, tort or otherwise shall be limited to the amounts paid by BNL to VIP in the twelve (12) month period prior to the accrual of the action or claim for the specific service which is the subject of the action or claim (or, if such accrual occurs during the first twelve (12) months of the initial term, then the liability shall be limited to the minimum fees payable by BNL to VIP during the first twelve (12) months of the initial term) claims by VIP for the minimum fees and other fees and expenses owing by BNL under paragraphs 5, 15(a) and 15(c), or for a breach by BNL of VIP's proprietary rights as set forth in paragraph 13 are excluded from this paragraph II limitation except for the claims excluded by the preceding sentence, this limitation applies

In [ ]:
# Diagnostic cell to check raw similarity scores
print(f"Max similarity found in last batch: {similarities.max().item():.4f}")
print(f"Average similarity found in last batch: {similarities.mean().item():.4f}")

# Let's see how many matches exist at various lower thresholds
for test_t in [0.40, 0.45, 0.50, 0.55]:
    count = (similarities >= test_t).sum().item()
    print(f"Pairs that would match at {test_t}: {count}")

Max similarity found in last batch: 0.5058
Average similarity found in last batch: 0.1963
Pairs that would match at 0.4: 552
Pairs that would match at 0.45: 67
Pairs that would match at 0.5: 1
Pairs that would match at 0.55: 0


Right now, we have a lot more judicial reasoning sentences that try to interpret the clauses than the actual clauses themselves. If we were to create a dataset using this, we would end up with a highly **imbalanced dataset.** However, there is something else that we can do, one that can highly speed up this process of constructing the triplet structure.

## Reverse Constructing the JDAR Triplet

Given that we already have the anchor clauses, i.e., the anchor clause, clause text, and the vague terms that were extracted from CUAD. Using these clause texts and the judical reasoning candidates that we have, we can use vector spaces to pair the clause texts with their judicial interpretation counterparts.

In essence, the triplet has the following structure:

$
\text{JDAR Triplet Data Point} = \begin{cases}
\textbf{Clause} : & \text{Clean CUAD Text Snippet} \\
\textbf{Terms} : & \text{Extracted Seed Words from the CUAD snippet and the judicial interpretation} \\
\textbf{Reasoning} : & \text{Aligned Harvard-LIL COLD Cases Opinion Sentence (i.e., the extracted reasoning sentences}
\end{cases}
$

In [ ]:
# prepare the structural pairs for cross attention processing
ce_pairs = [
    (item["cuad_anchor"], item["raw_sentence"])
    for item in distributed_threshold_results[threshold]
]

In [ ]:
from sentence_transformers import CrossEncoder

# Load the precision verification engine
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2').to("cuda")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# collate the candidates for the cross encoder
ce_pairs = []
candidate_metadata = []

In [ ]:
for row, col in zip(row_indices, column_indices):
  row_idx = row.item()
  col_idx = col.item()

  ce_pairs.append((clause_and_terms[row_idx]["clause_text"], window_string))

Passing 100 cadidiate pairs for Cross-Encoder verification............
Complete. Isolated 7 high-fidelity alignement pairs


In [ ]:
final_verified_triplets[4]

{'cuad_category': 'Post-Termination Services',
 'cuad_anchor': 'Upon the termination of this Agreement by either party:',
 'raw_sentence': '8\n\x0c     Case: 15-10881       Document: 00513863253         Page: 9    Date Filed: 02/03/2017\n\n\n                                      No. the Advisor shall not be entitled\nto compensation for further services hereunder but shall be paid all\ncompensation accruing to the date of expiration or termination.” To these eyes,\nthis clause makes it plain that the parties, at the time of contracting,\nenvisioned services—and payment for those services—continuing after notice\nwas given for the sixty days prior to the “effective date of . termination.” And\nwhile nothing in the Agreements bound the Publics to provide opportunities\nfor Prime to generate mortgage placement fees, acquisition commissions,\ndisposition fees, or loan arrangement fees during the notice period, 43 the\nAgreements did bind the Publics to pay the base compensation whether the